In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)

print("Ready")

Ready


In [3]:
# Loading datasets
reg = pd.read_csv('../datasets/beneficiaries_registration003.csv')
dist = pd.read_csv('../datasets/distribution_records004.csv')

print("Registration shape:", reg.shape)
print("Distribution shape:", dist.shape)

print("\nRegistration columns:", reg.columns.tolist())
print("Distribution columns:", dist.columns.tolist())

Registration shape: (20, 9)
Distribution shape: (20, 8)

Registration columns: ['beneficiary_id', 'full_name', 'age', 'gender', 'province', 'district', 'household_size', 'registration_date', 'phone']
Distribution columns: ['distribution_id', 'beneficiary_id', 'distribution_date', 'food_package_kg', 'enumerator', 'status', 'monthly_income_usd', 'vulnerability_score']


In [ ]:
# Check if beneficiary_id is unique in registration, nunique - count of unique values
print("Unique IDs in registration:", reg['beneficiary_id'].nunique())
print("Total rows in registration:", len(reg)) # if both equal every ID is unique

# Check if beneficiary_id is unique in distribution
print("\nUnique IDs in distribution:", dist['beneficiary_id'].nunique())
print("Total rows in distribution:", len(dist))

# Check which IDs are in distribution but not registration, set - to see the difference
dist_only = set(dist['beneficiary_id']) - set(reg['beneficiary_id'])
reg_only = set(reg['beneficiary_id']) - set(dist['beneficiary_id'])

print("\nIn distribution but not registration:", dist_only)
print("In registration but not distribution:", reg_only)

Unique IDs in registration: 20
Total rows in registration: 20

Unique IDs in distribution: 20
Total rows in distribution: 20

In distribution but not registration: {24, 21, 22, 23}
In registration but not distribution: {17, 11, 20, 6}


In [5]:
# Inner join —> only beneficiaries who are both registered AND received food
# only matching IDs (1-5, 7-10, 12-16, 18-19) = 16 rows

inner_merge = pd.merge(
    reg,
    dist,
    on='beneficiary_id',
    how='inner'
)

print("Inner join shape:", inner_merge.shape)
print("\nColumns:", inner_merge.columns.tolist())
print(inner_merge[['beneficiary_id', 'full_name', 'province', 'food_package_kg', 'status']])

Inner join shape: (16, 16)

Columns: ['beneficiary_id', 'full_name', 'age', 'gender', 'province', 'district', 'household_size', 'registration_date', 'phone', 'distribution_id', 'distribution_date', 'food_package_kg', 'enumerator', 'status', 'monthly_income_usd', 'vulnerability_score']
    beneficiary_id        full_name  province  food_package_kg     status
0                1     Ahmad Karimi     Kabul               25  Completed
1                2     Fatima Noori     Kabul               25  Completed
2                3   Mohammad Yusuf  Kandahar               25  Completed
3                4   Zainab Hussain     Herat               25  Completed
4                5      Abdul Karim     Kabul               25  Completed
5                7    Khalid Ahmadi  Kandahar               25  Completed
6                8     Sadia Rahimi     Kabul               25  Completed
7                9    Najiba Karimi     Balkh               25  Completed
8               10     Omar Sharifi     Kabul   

In [6]:
# loads tables to Postgres 
reg.to_sql('beneficiary_registration', engine, if_exists='replace', index=False)
dist.to_sql('distribution_records', engine, if_exists='replace', index=False)
print("Both tables loaded successfully")

Both tables loaded successfully


In [10]:
# SQL
# INNER JOIN ... ON —> connects two tables where the key matches. Only returns rows where match exists in both tables.
query = """
SELECT 
    r.beneficiary_id,
    r.full_name,
    r.province,
    r.household_size,
    d.food_package_kg,
    d.distribution_date,
    d.status,
    d.monthly_income_usd
FROM beneficiary_registration r
INNER JOIN distribution_records d
    ON r.beneficiary_id = d.beneficiary_id
ORDER BY r.beneficiary_id
"""
result = pd.read_sql(query, engine)
print(result)

    beneficiary_id        full_name  province  household_size  \
0                1     Ahmad Karimi     Kabul               6   
1                2     Fatima Noori     Kabul               4   
2                3   Mohammad Yusuf  Kandahar               3   
3                4   Zainab Hussain     Herat               5   
4                5      Abdul Karim     Kabul               7   
5                7    Khalid Ahmadi  Kandahar               9   
6                8     Sadia Rahimi     Kabul               2   
7                9    Najiba Karimi     Balkh               4   
8               10     Omar Sharifi     Kabul               6   
9               12  Habibullah Khan  Kandahar               8   
10              13    Razia Sultani     Herat               5   
11              14   Bismillah Omar     Kabul               7   
12              15     Laila Ahmadi     Balkh               2   
13              16     Qasim Wardak     Kabul               5   
14              18    Jaw